In [0]:
# ============================================
# CELL 1 — Install dependencies
# ============================================

%pip install imbalanced-learn xgboost --quiet

Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.


In [0]:
# ============================================
# CELL 2 — Imports
# ============================================

import mlflow
import mlflow.sklearn
import pandas as pd
import numpy as np

from sklearn.ensemble        import RandomForestClassifier
from sklearn.linear_model    import LogisticRegression
from xgboost                 import XGBClassifier
from sklearn.model_selection import train_test_split
from sklearn.preprocessing   import StandardScaler
from sklearn.metrics         import (
    classification_report,
    roc_auc_score,
    confusion_matrix,
    average_precision_score
)
from imblearn.over_sampling  import SMOTE

print("✅ All libraries imported successfully")

✅ All libraries imported successfully


In [0]:
# ============================================
# CELL 3 — Load Gold data
# ============================================

df_gold = spark.table("fraud_db.gold_features")
pdf     = df_gold.toPandas()

print(f"✅ Gold data loaded")
print(f"   Shape         : {pdf.shape}")
print(f"   Fraud cases   : {pdf['Class'].sum()}")
print(f"   Legit cases   : {(pdf['Class'] == 0).sum()}")
print(f"   Fraud rate    : {pdf['Class'].mean()*100:.4f}%")

✅ Gold data loaded
   Shape         : (281918, 46)
   Fraud cases   : 448
   Legit cases   : 281470
   Fraud rate    : 0.1589%


In [0]:
# ============================================
# CELL 4 — Prepare features
# ============================================

# Drop non-feature columns
drop_cols = [
    "ingestion_timestamp", "source_file",
    "pipeline_version", "silver_timestamp",
    "gold_timestamp", "amount_category",
    "amount_rounded"
]

feature_cols = [
    c for c in pdf.columns
    if c not in drop_cols + ["Class"]
]

X = pdf[feature_cols].fillna(0)
y = pdf["Class"]

print(f"✅ Features prepared")
print(f"   Number of features : {len(feature_cols)}")
print(f"   Feature list       : {feature_cols}")

✅ Features prepared
   Number of features : 38
   Feature list       : ['Time', 'V1', 'V2', 'V3', 'V4', 'V5', 'V6', 'V7', 'V8', 'V9', 'V10', 'V11', 'V12', 'V13', 'V14', 'V15', 'V16', 'V17', 'V18', 'V19', 'V20', 'V21', 'V22', 'V23', 'V24', 'V25', 'V26', 'V27', 'V28', 'Amount', 'is_high_value', 'hour_of_day', 'is_night', 'rolling_avg_amount', 'rolling_txn_count', 'rolling_fraud_rate', 'amount_vs_avg_ratio', 'is_amount_spike']


In [0]:
# ============================================
# CELL 5 — Split, scale & balance data
# ============================================

# Train/test split
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print(f"Train size : {X_train.shape[0]}")
print(f"Test size  : {X_test.shape[0]}")

# Scale features
scaler         = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled  = scaler.transform(X_test)

# Handle class imbalance with SMOTE
smote = SMOTE(random_state=42)
X_train_res, y_train_res = smote.fit_resample(
    X_train_scaled, y_train
)

print(f"\n✅ After SMOTE balancing:")
print(f"   Train size      : {X_train_res.shape[0]}")
print(f"   Fraud in train  : {sum(y_train_res)}")
print(f"   Legit in train  : {sum(y_train_res == 0)}")

Train size : 225534
Test size  : 56384

✅ After SMOTE balancing:
   Train size      : 450352
   Fraud in train  : 225176
   Legit in train  : 225176


In [0]:
# ============================================
# CELL 6 — Train all 3 models with MLflow
# (Fixed: added signature + input_example)
# ============================================

from mlflow.models.signature import infer_signature

mlflow.set_experiment("/fraud_detection_pipeline")

models = {
    "RandomForest": RandomForestClassifier(
        n_estimators=100,
        max_depth=10,
        class_weight="balanced",
        random_state=42,
        n_jobs=-1
    ),
    "XGBoost": XGBClassifier(
        n_estimators=100,
        max_depth=6,
        scale_pos_weight=576,
        random_state=42,
        eval_metric="logloss",
        use_label_encoder=False
    ),
    "LogisticRegression": LogisticRegression(
        class_weight="balanced",
        max_iter=1000,
        random_state=42
    )
}

best_auc    = 0
best_model  = None
best_name   = ""
results     = []

for model_name, model in models.items():
    print(f"\n⏳ Training {model_name}...")

    with mlflow.start_run(run_name=model_name):

        # Train
        model.fit(X_train_res, y_train_res)

        # Predict
        y_pred      = model.predict(X_test_scaled)
        y_pred_prob = model.predict_proba(
            X_test_scaled
        )[:, 1]

        # Metrics
        auc = roc_auc_score(y_test, y_pred_prob)
        ap  = average_precision_score(y_test, y_pred_prob)
        cm  = confusion_matrix(y_test, y_pred)
        tn, fp, fn, tp = cm.ravel()

        precision = tp / (tp + fp) if (tp + fp) > 0 else 0
        recall    = tp / (tp + fn) if (tp + fn) > 0 else 0
        f1 = (
            2 * precision * recall /
            (precision + recall)
            if (precision + recall) > 0 else 0
        )

        # ── Create signature (fixes Unity Catalog error) ──────
        input_example = pd.DataFrame(
            X_test_scaled[:5],
            columns=feature_cols
        )
        signature = infer_signature(
            input_example,
            model.predict(X_test_scaled[:5])
        )

        # Log to MLflow
        mlflow.log_param("model_type", model_name)
        mlflow.log_param("smote",      True)
        mlflow.log_param("scaler",     "StandardScaler")
        mlflow.log_metric("roc_auc",         round(auc, 4))
        mlflow.log_metric("avg_precision",   round(ap,  4))
        mlflow.log_metric("precision",       round(precision, 4))
        mlflow.log_metric("recall",          round(recall, 4))
        mlflow.log_metric("f1_score",        round(f1, 4))
        mlflow.log_metric("true_positives",  int(tp))
        mlflow.log_metric("false_positives", int(fp))
        mlflow.log_metric("false_negatives", int(fn))

        # Log model WITH signature and input example
        mlflow.sklearn.log_model(
            model,
            artifact_path=f"model_{model_name}",
            registered_model_name=f"fraud_{model_name}",
            signature=signature,
            input_example=input_example
        )

        # Store results
        results.append({
            "Model"    : model_name,
            "ROC-AUC"  : round(auc, 4),
            "Precision": round(precision, 4),
            "Recall"   : round(recall, 4),
            "F1-Score" : round(f1, 4),
            "TP"       : int(tp),
            "FP"       : int(fp),
            "FN"       : int(fn)
        })

        print(f"   ROC-AUC   : {auc:.4f}")
        print(f"   Precision : {precision:.4f}")
        print(f"   Recall    : {recall:.4f}")
        print(f"   F1-Score  : {f1:.4f}")
        print(f"   Fraud caught (TP) : {tp}")
        print(f"   Fraud missed (FN) : {fn}")
        print(f"   False alarms (FP) : {fp}")

        if auc > best_auc:
            best_auc   = auc
            best_model = model
            best_name  = model_name

# ── Results comparison table ─────────────────────────────────────
print("\n")
print("=" * 65)
print("           MODEL COMPARISON RESULTS")
print("=" * 65)
df_results = pd.DataFrame(results)
print(df_results.to_string(index=False))
print("=" * 65)
print(f"\n🏆 Best model : {best_name}")
print(f"   ROC-AUC   : {best_auc:.4f}")


⏳ Training RandomForest...


/databricks/python/lib/python3.12/site-packages/sklearn/utils/validation.py:2732: UserWarning: X has feature names, but RandomForestClassifier was fitted without feature names
  warnings.warn(
Registered model 'fraud_RandomForest' already exists. Creating a new version of this model...
Created version '1' of model 'workspace.default.fraud_randomforest'.


   ROC-AUC   : 0.9994
   Precision : 0.3871
   Recall    : 0.9333
   F1-Score  : 0.5472
   Fraud caught (TP) : 84
   Fraud missed (FN) : 6
   False alarms (FP) : 133

⏳ Training XGBoost...


/local_disk0/.ephemeral_nfs/envs/pythonEnv-e3c5d2d4-6c44-45d9-86dc-b628bf011efb/lib/python3.12/site-packages/xgboost/training.py:200: UserWarning: [14:18:52] WARNING: /__w/xgboost/xgboost/src/learner.cc:782: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)
Successfully registered model 'workspace.default.fraud_xgboost'.
Created version '1' of model 'workspace.default.fraud_xgboost'.


   ROC-AUC   : 0.9892
   Precision : 0.6406
   Recall    : 0.9111
   F1-Score  : 0.7523
   Fraud caught (TP) : 82
   Fraud missed (FN) : 8
   False alarms (FP) : 46

⏳ Training LogisticRegression...


/databricks/python/lib/python3.12/site-packages/sklearn/utils/validation.py:2732: UserWarning: X has feature names, but LogisticRegression was fitted without feature names
  warnings.warn(
Successfully registered model 'workspace.default.fraud_logisticregression'.


   ROC-AUC   : 0.9978
   Precision : 0.0552
   Recall    : 0.9889
   F1-Score  : 0.1045
   Fraud caught (TP) : 89
   Fraud missed (FN) : 1
   False alarms (FP) : 1524


           MODEL COMPARISON RESULTS
             Model  ROC-AUC  Precision  Recall  F1-Score  TP   FP  FN
      RandomForest   0.9994     0.3871  0.9333    0.5472  84  133   6
           XGBoost   0.9892     0.6406  0.9111    0.7523  82   46   8
LogisticRegression   0.9978     0.0552  0.9889    0.1045  89 1524   1

🏆 Best model : RandomForest
   ROC-AUC   : 0.9994


Created version '1' of model 'workspace.default.fraud_logisticregression'.


In [0]:
# Diagnose the issue
print(f"Best model name : {best_name}")
print(f"Best model      : {best_model}")
print(f"Best AUC        : {best_auc}")
print(f"Results table   :")
print(pd.DataFrame(results))

Best model name : RandomForest
Best model      : RandomForestClassifier(class_weight='balanced', max_depth=10, n_jobs=-1,
                       random_state=42)
Best AUC        : 0.9993802378781239
Results table   :
                Model  ROC-AUC  Precision  Recall  F1-Score  TP    FP  FN
0        RandomForest   0.9994     0.3871  0.9333    0.5472  84   133   6
1             XGBoost   0.9892     0.6406  0.9111    0.7523  82    46   8
2  LogisticRegression   0.9978     0.0552  0.9889    0.1045  89  1524   1


In [0]:
# ============================================
# CELL 7 — Save predictions to Delta
# (Fixed: safely retrain best model if needed)
# ============================================

# ── Step 1: Identify best model from results ─────────────────────
df_results = pd.DataFrame(results)
best_name  = df_results.loc[
    df_results["ROC-AUC"].idxmax(), "Model"
]
print(f"🏆 Best model identified: {best_name}")

# ── Step 2: Retrain best model cleanly ───────────────────────────
best_models_map = {
    "RandomForest": RandomForestClassifier(
        n_estimators=100,
        max_depth=10,
        class_weight="balanced",
        random_state=42,
        n_jobs=-1
    ),
    "XGBoost": XGBClassifier(
        n_estimators=100,
        max_depth=6,
        scale_pos_weight=576,
        random_state=42,
        eval_metric="logloss",
        use_label_encoder=False
    ),
    "LogisticRegression": LogisticRegression(
        class_weight="balanced",
        max_iter=1000,
        random_state=42
    )
}

best_model = best_models_map[best_name]
print(f"⏳ Retraining {best_name} for predictions...")
best_model.fit(X_train_res, y_train_res)
print(f"✅ {best_name} retrained successfully")

# ── Step 3: Generate predictions ─────────────────────────────────
y_pred_final = best_model.predict(X_test_scaled)
y_prob_final = best_model.predict_proba(
    X_test_scaled
)[:, 1]

# ── Step 4: Build predictions dataframe ──────────────────────────
df_test = pd.DataFrame(
    X_test.values,
    columns=feature_cols
)
df_test["actual_class"]      = y_test.values
df_test["predicted_class"]   = y_pred_final
df_test["fraud_probability"] = y_prob_final

# ── Step 5: Add risk level ────────────────────────────────────────
def get_risk_level(prob):
    if prob >= 0.90:
        return "CRITICAL"
    elif prob >= 0.70:
        return "HIGH"
    elif prob >= 0.50:
        return "MEDIUM"
    else:
        return "LOW"

df_test["risk_level"] = df_test[
    "fraud_probability"
].apply(get_risk_level)

# ── Step 6: Preview results ───────────────────────────────────────
print("\n=== PREDICTION SUMMARY ===")
print(f"Total predictions  : {len(df_test)}")
print(f"Fraud predicted    : {df_test['predicted_class'].sum()}")
print(f"Critical risk      : {(df_test['risk_level']=='CRITICAL').sum()}")
print(f"High risk          : {(df_test['risk_level']=='HIGH').sum()}")
print(f"Medium risk        : {(df_test['risk_level']=='MEDIUM').sum()}")
print(f"Low risk           : {(df_test['risk_level']=='LOW').sum()}")

print("\n=== TOP 10 HIGHEST RISK TRANSACTIONS ===")
print(
    df_test.nlargest(10, "fraud_probability")[[
        "Amount", "actual_class",
        "predicted_class", "fraud_probability",
        "risk_level"
    ]].to_string(index=False)
)

# ── Step 7: Save to Delta ─────────────────────────────────────────
spark.createDataFrame(df_test) \
    .write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("fraud_db.gold_predictions")

print(f"\n✅ Predictions saved: fraud_db.gold_predictions")
print(f"   Best model used : {best_name}")
print(f"   Rows written    : {len(df_test)}")

🏆 Best model identified: RandomForest
⏳ Retraining RandomForest for predictions...
✅ RandomForest retrained successfully

=== PREDICTION SUMMARY ===
Total predictions  : 56384
Fraud predicted    : 217
Critical risk      : 77
High risk          : 22
Medium risk        : 118
Low risk           : 56167

=== TOP 10 HIGHEST RISK TRANSACTIONS ===
 Amount  actual_class  predicted_class  fraud_probability risk_level
   2.28             1                1           0.999820   CRITICAL
   1.00             1                1           0.999816   CRITICAL
  99.85             1                1           0.999814   CRITICAL
 209.65             1                1           0.999813   CRITICAL
  75.86             1                1           0.999808   CRITICAL
  53.95             1                1           0.999806   CRITICAL
 316.06             1                1           0.999803   CRITICAL
   1.00             1                1           0.999803   CRITICAL
  37.32             1               